# Detect Font Sizes in PDF Using PyMuPDF and Draw Boxes with OpenCV

This notebook uses **PyMuPDF** to extract font sizes and bounding boxes of text spans from a PDF, and **OpenCV** to draw rectangles around those spans with font size annotations.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# You'll need to install these packages:
# pip install PyMuPDF opencv-python matplotlib numpy

try:
    import fitz  # PyMuPDF
except ImportError:
    print("Please install PyMuPDF: pip install PyMuPDF")
    exit(1)

def detect_squares_in_pdf(pdf_path, page_num=0, dpi=300, min_area=50, max_area=50000):
    """
    Detects and highlights all square-like shapes in a PDF page.
    
    Args:
        pdf_path (str): Path to the PDF file
        page_num (int): Page number to analyze (0-indexed)
        dpi (int): Resolution for rendering the PDF page
        min_area (int): Minimum area threshold for square detection
        max_area (int): Maximum area threshold for square detection
    """
    
    # Open PDF
    try:
        doc = fitz.open(pdf_path)
        print(f"Successfully opened PDF with {len(doc)} pages")
    except Exception as e:
        print(f"Error opening PDF file: {e}")
        return

    if page_num >= len(doc):
        print(f"Error: Page number {page_num} is out of range.")
        doc.close()
        return

    page = doc[page_num]
    
    # Convert PDF page to image
    pix = page.get_pixmap(dpi=dpi)
    img_data = np.frombuffer(pix.samples, dtype=np.uint8)
    img = img_data.reshape(pix.h, pix.w, pix.n)
    
    # Handle different color formats
    if pix.n == 4:  # RGBA
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    elif pix.n == 3:  # RGB
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    else:  # Grayscale
        img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # Convert to grayscale for processing
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    # Apply multiple preprocessing techniques to catch different types of squares
    detected_squares = []
    
    # Method 1: Standard thresholding
    _, thresh1 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    squares1 = find_squares_in_binary(thresh1, min_area, max_area)
    detected_squares.extend(squares1)
    
    # Method 2: Adaptive thresholding
    thresh2 = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY, 11, 2)
    squares2 = find_squares_in_binary(thresh2, min_area, max_area)
    detected_squares.extend(squares2)
    
    # Method 3: Inverted binary
    _, thresh3 = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    squares3 = find_squares_in_binary(thresh3, min_area, max_area)
    detected_squares.extend(squares3)
    
    # Method 4: Edge detection approach
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    squares4 = find_squares_in_binary(edges, min_area, max_area)
    detected_squares.extend(squares4)
    
    # Remove duplicate detections
    detected_squares = remove_duplicate_squares(detected_squares)
    
    print(f"Found {len(detected_squares)} potential squares")
    
    # Visualize results
    visualize_results(img_bgr, detected_squares, page_num)
    
    doc.close()
    return detected_squares

def find_squares_in_binary(binary_img, min_area, max_area):
    """Find square contours in a binary image."""
    squares = []
    
    # Find contours
    contours, _ = cv2.findContours(binary_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    for contour in contours:
        # Approximate contour to polygon
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        # Check if it's a quadrilateral
        if len(approx) == 4:
            # Get bounding rectangle
            x, y, w, h = cv2.boundingRect(approx)
            
            # Calculate area
            area = cv2.contourArea(contour)
            
            # Check if it's square-like (aspect ratio close to 1)
            aspect_ratio = w / float(h) if h != 0 else 0
            
            # Check if contour is convex (squares should be convex)
            is_convex = cv2.isContourConvex(approx)
            
            # Apply filters
            if (0.8 <= aspect_ratio <= 1.2 and  # Square-like aspect ratio
                min_area <= area <= max_area and   # Size constraints
                is_convex):                        # Must be convex
                
                squares.append((x, y, w, h, area))
    
    return squares

def remove_duplicate_squares(squares, overlap_threshold=0.7):
    """Remove overlapping square detections."""
    if not squares:
        return []
    
    # Sort by area (largest first)
    squares = sorted(squares, key=lambda x: x[4], reverse=True)
    unique_squares = []
    
    for current in squares:
        x1, y1, w1, h1, _ = current
        is_duplicate = False
        
        for existing in unique_squares:
            x2, y2, w2, h2, _ = existing
            
            # Calculate intersection over union (IoU)
            intersection_x = max(0, min(x1 + w1, x2 + w2) - max(x1, x2))
            intersection_y = max(0, min(y1 + h1, y2 + h2) - max(y1, y2))
            intersection_area = intersection_x * intersection_y
            
            union_area = w1 * h1 + w2 * h2 - intersection_area
            
            if union_area > 0:
                iou = intersection_area / union_area
                if iou > overlap_threshold:
                    is_duplicate = True
                    break
        
        if not is_duplicate:
            unique_squares.append(current)
    
    return [(x, y, w, h) for x, y, w, h, _ in unique_squares]

def visualize_results(img, squares, page_num):
    """Display the image with highlighted squares."""
    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(15, 20))
    ax.imshow(img_rgb)
    ax.set_title(f"Detected Squares on Page {page_num + 1} (Found: {len(squares)})", 
                fontsize=16, fontweight='bold')
    
    # Draw rectangles around detected squares
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan']
    for i, (x, y, w, h) in enumerate(squares):
        color = colors[i % len(colors)]
        rect = Rectangle((x, y), w, h, linewidth=3, edgecolor=color, 
                        facecolor='none', alpha=0.8)
        ax.add_patch(rect)
        
    
    
    
    ax.axis('off')
    plt.tight_layout()
    plt.show()

def main():
    # Example usage
    pdf_path = "Claim_Form.pdf"  # Replace with your PDF path
    page_num = 0  # First page (0-indexed)
    
    print("Starting square detection...")
    squares = detect_squares_in_pdf(
        pdf_path=pdf_path,
        page_num=page_num,
        dpi=300,           # Higher DPI = better quality but slower
        min_area=50,       # Minimum square size in pixels
        max_area=50000     # Maximum square size in pixels
    )
    
    if squares:
        print("\nDetected squares:")
        for i, (x, y, w, h) in enumerate(squares):
            print(f"  Square {i+1}: Position({x}, {y}), Size({w}x{h})")
    else:
        print("No squares detected.")

if __name__ == "__main__":
    main()


In [ ]:
def detect_isolated_squares_in_pdf(pdf_path, page_num=0, isolation_distance=60):
    """
    Complete pipeline to detect only isolated squares in a PDF.
    
    Args:
        pdf_path (str): Path to PDF file
        page_num (int): Page number (0-indexed)
        isolation_distance (int): Minimum distance for isolation
    
    Returns:
        list: Isolated squares only
    """
    # Open PDF and convert to image
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(dpi=300)
    
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    if pix.n == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    
    # Detect all squares first (using previous detection logic)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    all_squares = []
    
    for contour in contours:
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        if len(approx) == 4 and cv2.isContourConvex(approx):
            x, y, w, h = cv2.boundingRect(approx)
            aspect_ratio = w / float(h) if h != 0 else 0
            area = cv2.contourArea(contour)
            
            if 0.8 <= aspect_ratio <= 1.2 and 100 <= area <= 10000:
                all_squares.append((x, y, w, h))
    
    # Filter for isolated squares only
    isolated_squares = filter_isolated_squares(all_squares, isolation_distance)
    
    # Visualize results
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # All squares
    ax1.imshow(img_rgb)
    ax1.set_title(f"All Detected Squares ({len(all_squares)})")
    for x, y, w, h in all_squares:
        rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='blue', facecolor='none')
        ax1.add_patch(rect)
    ax1.axis('off')
    
    # Isolated squares only
    ax2.imshow(img_rgb)
    ax2.set_title(f"Isolated Squares Only ({len(isolated_squares)})")
    for x, y, w, h in isolated_squares:
        rect = Rectangle((x, y), w, h, linewidth=3, edgecolor='red', facecolor='none')
        ax2.add_patch(rect)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    doc.close()
    return isolated_squares

# Usage examples:
isolated = detect_isolated_squares_in_pdf("./IRIS/checkbox_trials/Claim_Form.pdf", page_num=0, isolation_distance=50)
# 
# # Or use individual functions:
# all_squares = detect_all_squares(pdf_path)  # Your existing detection
# isolated_only = filter_isolated_squares(all_squares, distance_threshold=60)
# categorized = context_aware_isolation(isolated_only, page_width, page_height)

In [ ]:
def filter_isolated_squares(squares, distance_threshold=50, method='euclidean'):
    """
    Filters squares that have no other squares within a specified distance.
    
    Args:
        squares (list): List of tuples (x, y, w, h) representing detected squares
        distance_threshold (int): Minimum distance to consider squares as neighbors
        method (str): Distance calculation method - 'euclidean', 'manhattan', or 'grid'
    
    Returns:
        list: Filtered list of isolated squares
    """
    isolated_squares = []
    
    for i, (x1, y1, w1, h1) in enumerate(squares):
        # Calculate center point of current square
        center1_x = x1 + w1 / 2
        center1_y = y1 + h1 / 2
        
        is_isolated = True
        
        # Check distance to all other squares
        for j, (x2, y2, w2, h2) in enumerate(squares):
            if i == j:  # Skip self-comparison
                continue
                
            # Calculate center of comparison square
            center2_x = x2 + w2 / 2
            center2_y = y2 + h2 / 2
            
            # Calculate distance based on method
            if method == 'euclidean':
                distance = ((center1_x - center2_x)**2 + (center1_y - center2_y)**2)**0.5
                if distance < distance_threshold:
                    is_isolated = False
                    break
                    
            elif method == 'manhattan':
                distance = abs(center1_x - center2_x) + abs(center1_y - center2_y)
                if distance < distance_threshold:
                    is_isolated = False
                    break
                    
            elif method == 'grid':
                # Grid-based: check if squares are aligned horizontally or vertically
                dist_x = abs(center1_x - center2_x)
                dist_y = abs(center1_y - center2_y)
                
                # Consider as neighbors if close in either direction
                if (dist_x < distance_threshold and dist_y < distance_threshold * 2) or \
                   (dist_y < distance_threshold and dist_x < distance_threshold * 2):
                    is_isolated = False
                    break
        
        if is_isolated:
            isolated_squares.append((x1, y1, w1, h1))
    
    return isolated_squares


In [ ]:

"""
Jupyter Notebook script to visualize the output of the IRIS Perception Layer.

This script uses ONLY the PyMuPDF (fitz) library for all PDF operations,
including rendering the page to an image.

1. Text Blob Detection: Uses a 'squinting' (dilation) technique to find the
   boundaries of visually grouped blocks of text. These are drawn in GREEN.
2. Indicator Shape Detection: Finds shapes that indicate input fields, such as
   boxes (RED), checkboxes (BLUE), and underlines (PURPLE).

The final output is an image of the PDF page with all detected boundaries drawn on it.
"""

# --- Step 1: Imports and Configuration ---
import os
import cv2
import fitz  # PyMuPDF
import numpy as np
import matplotlib.pyplot as plt

# --- Configuration ---
PDF_PATH = "Claim_Form.pdf"
PAGE_NUM = 0  # Process the first page (index 0)
DPI = 900    # Dots per inch for rendering the PDF page to an image

# --- Step 2: Helper Functions (Core Logic) ---
# (These helper functions remain unchanged from the previous version)

def detect_text_blobs(image, scale_x=1.0, scale_y=1.0):
    """
    Uses a CV 'squint' method (dilation) to find bounding boxes of text blocks.
    """
    print("   - Grayscaling and applying binary threshold...")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]

    kernel_height = 5
    kernel_width = 40
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_width, kernel_height))
    
    print("   - Dilating image to merge text blobs...")
    dilated = cv2.dilate(binary, kernel, iterations=4)

    print("   - Finding contours of text blobs...")
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    blob_boundaries = []
    MIN_CONTOUR_AREA = 1000
    for contour in contours:
        if cv2.contourArea(contour) > MIN_CONTOUR_AREA:
            x, y, w, h = cv2.boundingRect(contour)
            x0, y0 = x * scale_x, y * scale_y
            x1, y1 = (x + w) * scale_x, (y + h) * scale_y
            blob_boundaries.append([round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)])
            
    return blob_boundaries

def detect_indicator_shapes(image, scale_x=1.0, scale_y=1.0):
    """
    Detects shapes like underlines, boxes, and checkboxes that indicate input fields.
    """
    print("   - Detecting indicator shapes (boxes, checkboxes)...")
    indicator_shapes = []
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 11, 2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    MIN_CHECKBOX_SIZE = 8
    MAX_CHECKBOX_SIZE = 25
    ASPECT_RATIO_TOLERANCE = 0.4

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = w / float(h) if h > 0 else 0
        
        is_checkbox_size = MIN_CHECKBOX_SIZE < w < MAX_CHECKBOX_SIZE and MIN_CHECKBOX_SIZE < h < MAX_CHECKBOX_SIZE
        is_square_like = abs(1 - aspect_ratio) < ASPECT_RATIO_TOLERANCE
        is_underline = aspect_ratio > 10 and h < 10
        is_box = (w > 15 and h > 15) and (0.5 < aspect_ratio < 10)

        shape_type = None
        if is_checkbox_size and is_square_like:
            shape_type = "CHECKBOX"
        elif is_underline:
            shape_type = "UNDERLINE"
        elif is_box:
            shape_type = "BOX"

        if shape_type:
            x0, y0 = x * scale_x, y * scale_y
            x1, y1 = (x + w) * scale_x, (y + h) * scale_y
            shape = {
                "type": shape_type,
                "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)]
            }
            indicator_shapes.append(shape)
            
    return indicator_shapes


# --- Step 3: Main Processing and Visualization ---

def run_perception_and_visualize():
    """Main function to run the pipeline and display the results."""
    
    if not os.path.exists(PDF_PATH):
        print(f"❌ ERROR: File not found at '{PDF_PATH}'")
        return

    # --- 1. Load PDF and Render Page using PyMuPDF ---
    print(f"📖 Loading PDF with PyMuPDF and rendering page {PAGE_NUM}...")
    doc = fitz.open(PDF_PATH)
    page = doc[PAGE_NUM]
    
    # Render page to a pixmap (an image object in PyMuPDF)
    pix = page.get_pixmap(dpi=DPI)
    
    # Convert pixmap to an OpenCV-compatible NumPy array
    # pix.samples contains the raw image data (e.g., RGB, RGBA)
    img_data = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    
    # Convert to BGR format for OpenCV
    if pix.n == 4: # RGBA
        page_image_cv = cv2.cvtColor(img_data, cv2.COLOR_RGBA2BGR)
    elif pix.n == 3: # RGB
        page_image_cv = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
    else: # Grayscale or other, convert to BGR
        page_image_cv = cv2.cvtColor(img_data, cv2.COLOR_GRAY2BGR)

    drawing_image = page_image_cv.copy()
    
    # --- 2. Calculate coordinate scaling factors ---
    print("📏 Calculating coordinate scaling factors...")
    img_h, img_w, _ = page_image_cv.shape
    scale_x = page.rect.width / img_w
    scale_y = page.rect.height / img_h

    # --- 3. Run Detection Algorithms ---
    print("\n🧠 Running Perception Algorithms...")
    text_blob_bboxes = detect_text_blobs(page_image_cv, scale_x, scale_y)
    print(f"   ✅ Found {len(text_blob_bboxes)} text blobs.")
    
    indicator_shapes = detect_indicator_shapes(page_image_cv, scale_x, scale_y)
    print(f"   ✅ Found {len(indicator_shapes)} indicator shapes.")
    
    doc.close() # Close the document after use

    # --- 4. Draw Boundaries for Visualization ---
    print("\n🎨 Drawing detected boundaries on the image...")
    COLOR_TEXT_BLOB = (0, 255, 0)      # Green
    COLOR_BOX = (0, 0, 255)            # Red
    COLOR_CHECKBOX = (255, 0, 0)       # Blue
    COLOR_UNDERLINE = (218, 112, 214)  # Orchid/Purple

    for bbox in text_blob_bboxes:
        # Convert from PDF coordinates back to image coordinates for drawing
        x0, y0, x1, y1 = [int(c / scale_x) if i % 2 == 0 else int(c / scale_y) for i, c in enumerate(bbox)]
        cv2.rectangle(drawing_image, (x0, y0), (x1, y1), COLOR_TEXT_BLOB, 3)

    for shape in indicator_shapes:
        bbox = shape["bbox"]
        x0, y0, x1, y1 = [int(c / scale_x) if i % 2 == 0 else int(c / scale_y) for i, c in enumerate(bbox)]
        color = COLOR_BOX
        if shape["type"] == "CHECKBOX": color = COLOR_CHECKBOX
        elif shape["type"] == "UNDERLINE": color = COLOR_UNDERLINE
        cv2.rectangle(drawing_image, (x0, y0), (x1, y1), color, 2)

    # --- 5. Display the Final Image ---
    print("✨ Displaying final results...")
    plt.figure(figsize=(20, 30))
    plt.imshow(cv2.cvtColor(drawing_image, cv2.COLOR_BGR2RGB))
    plt.title("IRIS Perception Layer Output (Pure PyMuPDF)\n(Green: Text Blobs, Red: Input Boxes, Blue: Checkboxes)", fontsize=20)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# --- Run the script ---
if __name__ == "__main__":
    run_perception_and_visualize()